In [ ]:
import torch
import seaborn as sns
from torch.utils.data import Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=256):
        self.texts      = texts
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text   = str(self.texts[idx])
        inputs = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            'input_ids':      inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
        }

# 3. Định nghĩa lớp SentimentDataset có kế thừa
class SentimentDataset(TextDataset):
    def __init__(self, texts, tokenizer, labels=None, max_length=256):
        super().__init__(texts, tokenizer, max_length=max_length)
        self.labels = labels

    def __getitem__(self, idx):
        item = super().__getitem__(idx)
        # Đưa các tensor lên đúng thiết bị phần cứng (GPU/CPU)
        item = {k: v.to(DEVICE) for k, v in item.items()}
        
        # Tích hợp thêm cấu trúc nhãn long tensor cho bài toán phân loại cảm xúc
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long).to(DEVICE)
        return item

print(" Định nghĩa cấu trúc SentimentDataset thành công!")

 Định nghĩa cấu trúc SentimentDataset thành công!


In [ ]:
import pandas as pd
from transformers import AutoTokenizer

train_df = pd.read_csv("../data/processed/train_cleaned.csv").dropna()
test_df  = pd.read_csv("../data/raw/test.csv").dropna()

#  Khởi tạo Tokenizer PhoBERT
tokenizer = AutoTokenizer.from_pretrained("vinai/phobert-base")

# Tạo biến train_dataset và test_dataset với tên cột 'tokenized' và 'comment' chuẩn của nhóm
MAX_LENGTH = 256

train_dataset = SentimentDataset(
    texts=train_df['tokenized'].values,
    labels=train_df['label'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

test_dataset = SentimentDataset(
    texts=test_df['comment'].values,
    labels=test_df['label'].values,
    tokenizer=tokenizer,
    max_length=MAX_LENGTH
)

print(f" Đọc file và khởi tạo Dataset thành công!")
print(f"Số mẫu tập Train: {len(train_dataset):,} | Số mẫu tập Test: {len(test_dataset):,}")

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import classification_report, confusion_matrix

train_loader = DataLoader(
    train_dataset, 
    batch_size=32, 
    shuffle=True
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=16, 
    shuffle=False
)

# 1. Cấu hình thiết bị phần cứng (Ưu tiên GPU)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Thiết bị huấn luyện chính: {DEVICE}")
if torch.cuda.is_available():
    print(f"Cấu hình GPU hệ thống: {torch.cuda.get_device_name(0)}")

# 2. Khởi tạo mô hình học sâu PhoBERT Sequence Classification   
MODEL_NAME = "vinai/phobert-base"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(DEVICE) # Đẩy toàn bộ mạng nơ-ron lên cấu hình GPU

print(" Khởi tạo cấu trúc PhoBERT và đẩy lên thiết bị huấn luyện thành công.")

In [ ]:
# 1. Định nghĩa tham số huấn luyện cơ bản 
EPOCHS = 5 
LEARNING_RATE = 1e-5  # Thử nghiệm dải học nhỏ hơn (1e-5 hoặc 3e-5) để mịn đường cong loss

# 2. Tính toán trọng số lớp thực tế từ phân bố nhãn tập Train [5268, 491, 5667]
class_samples = np.array([5268, 491, 5667])
total_samples = class_samples.sum()
class_weights = total_samples / (len(class_samples) * class_samples)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)
print(f"Mảng trọng số phạt lỗi được tối ưu: {class_weights}")

# 3. Khởi tạo hàm loss tích hợp trọng số phạt lỗi diện rộng chống mất cân bằng nhãn
loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)

# Tính tổng số bước huấn luyện phục vụ bộ giảm học vị tuyến tính
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps), 
    num_training_steps=total_steps
)
print(" Cấu hình tham số và bộ tối ưu hóa thành công!")

In [ ]:
train_loss_history = []
val_loss_history = []

for epoch in range(EPOCHS):
    print(f"\n========== EPOCH {epoch + 1} / {EPOCHS} ==========")
    
    # ---------------- TRẠNG THÁI HUẤN LUYỆN (TRAINING) ----------------
    model.train()
    total_train_loss = 0
    
    for step, batch in enumerate(train_loader):
        # Đảm bảo ép thủ công toàn bộ các đầu vào Tensor lên cùng Device cấu hình GPU
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        
        # Reset các gradient về 0 trước mỗi bước tính toán
        model.zero_grad()
        
        # Đẩy dữ liệu qua mạng PhoBERT
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        
        # Tính toán hàm loss tích hợp mảng trọng số phạt lỗi chống mất cân bằng nhãn
        loss = loss_fn(logits, labels)
        total_train_loss += loss.item()
        
        # Lan truyền ngược (Backpropagation)
        loss.backward()
        
        # Thực hiện kỹ thuật cắt xén gradient (Gradient Clipping) tránh bùng nổ gradient
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        # Cập nhật tham số trọng số và điều chỉnh tốc độ học
        optimizer.step()
        scheduler.step()
        
        if step % 50 == 0 and step > 0:
            print(f"  Bước {step}/{len(train_loader)}  |  Loss huấn luyện hiện tại: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_loader)
    train_loss_history.append(avg_train_loss)
    print(f"Trung bình Loss Huấn luyện của Epoch {epoch + 1}: {avg_train_loss:.4f}")
    
    # ---------------- TRẠNG THÁI ĐÁNH GIÁ (EVALUATION) ----------------
    model.eval()
    total_val_loss = 0
    
    # Sử dụng torch.no_grad() để đóng băng gradient, tiết kiệm tài nguyên GPU
    with torch.no_grad():
        for batch in test_loader: # Dùng test_loader làm mốc validation kiểm soát loss
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels = batch['labels'].to(DEVICE)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            
            loss = loss_fn(logits, labels)
            total_val_loss += loss.item()
            
    avg_val_loss = total_val_loss / len(test_loader)
    val_history.append(avg_val_loss) if 'val_history' in locals() else val_loss_history.append(avg_val_loss)
    print(f"Trung bình Loss Kiểm thử của Epoch {epoch + 1}: {avg_val_loss:.4f}")

print("\n Quá trình Fine-tuning PhoBERT hoàn tất.")

# --- VẼ BIỂU ĐỒ ĐƯỜNG CONG HỘI TỤ LOSS CURVE ---
plt.figure(figsize=(10, 6))
plt.plot(range(1, EPOCHS + 1), train_loss_history, label='Loss Tập Huấn Luyện (Train Set)', marker='o')
plt.plot(range(1, EPOCHS + 1), val_loss_history, label='Loss Tập Kiểm Thử (Test Set)', marker='s')
plt.title('Biểu đồ đường cong hội tụ kiểm soát Overfitting — Tuần 4')
plt.xlabel('Epochs')
plt.ylabel('Loss Value')
plt.legend()
plt.grid(True)
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/loss_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# 1. Trích xuất dự đoán thực tế từ mô hình trên tập Test ẩn hoàn toàn
model.eval()
all_preds = []
all_labels = []
val_predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        val_predictions.extend(outputs.logits.cpu().numpy())
        preds = torch.argmax(outputs.logits, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# 2. Xuất bảng dữ liệu chỉ số Metric học thuật ra file CSV bàn giao cho Huy
report_dict = classification_report(all_labels, all_preds, output_dict=True)
df_metrics = pd.DataFrame(report_dict).transpose()
df_metrics.to_csv('../results/final_metrics.csv', index=True)

print("BÁO CÁO HIỆU NĂNG PHÂN LOẠI TỐI HẬU PHOBERT (TUẦN 4):")
print(classification_report(all_labels, all_preds, target_names=['Tiêu cực (0)', 'Trung tính (1)', 'Tích cực (2)']))

# 3. Vẽ và lưu trực quan hóa Ma Trận Nhầm Lẫn
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Dự đoán 0', 'Dự đoán 1', 'Dự đoán 2'],
            yticklabels=['Nhãn thật 0', 'Nhãn thật 1', 'Nhãn thật 2'])
plt.title('Ma trận nhầm lẫn tối hậu — Mô hình học sâu PhoBERT')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('../results/phobert_confusion_matrix.png', bbox_inches='tight')
plt.show()

# 4. LƯU LẠI FILE TRỌNG SỐ MÔ HÌNH (BÀI BÀN GIAO CHO KIỆT LÀM DEMO)
MODEL_SAVE_DIR = "../model_checkpoints"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
model.save_pretrained(MODEL_SAVE_DIR)
print(f" Đã đóng gói checkpoint cấu hình trọng số thành công tại: {MODEL_SAVE_DIR}")

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

print("--- TIẾN HÀNH TRÍCH XUẤT XÁC SUẤT MỀM VÀ TÌM NGƯỠNG TỐI ƯU ---")

# 1. Chuyển đổi list chứa logits thô từ quá trình đánh giá (val_predictions) thành mảng NumPy
# (Đảm bảo biến val_predictions hoặc test_predictions của Tài đang lưu logits đầu ra mô hình)
val_logits = np.array(val_predictions) 
y_true = np.array(all_labels) # Nhãn thật của tập test (3,166 mẫu)

# 2. Áp dụng hàm toán học Softmax để quy đổi Logits sang xác suất mềm [0, 1]
exp_logits = np.exp(val_logits - np.max(val_logits, axis=-1, keepdims=True))
y_probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

best_threshold = 0.5
best_macro_f1 = 0.0
final_preds = []

# 3. Thuật toán quét dải ngưỡng linh hoạt từ 0.20 đến 0.60
for threshold in np.arange(0.2, 0.6, 0.02):
    current_preds = []
    for prob in y_probs:
        # Nếu xác suất nhãn 1 (Trung tính) vượt qua ngưỡng đang quét -> Ưu tiên ép gán về nhãn 1
        if prob[1] >= threshold:
            current_preds.append(1)
        else:
            # Ngược lại, so sánh xác suất để chọn nhãn cao hơn giữa nhãn 0 và nhãn 2
            current_preds.append(0 if prob[0] > prob[2] else 2)
            
    # Tính toán Macro-F1 tại ngưỡng hiện tại
    rep = classification_report(y_true, current_preds, output_dict=True)
    macro_f1 = rep['macro avg']['f1-score']
    
    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_threshold = threshold
        final_preds = current_preds

print(f"\n[KẾT QUẢ TINH CHỈNH NGƯỠNG - TUẦN 5]")
print(f"-> Ngưỡng quyết định tối ưu cho lớp Trung tính: {best_threshold:.2f}")
print(f"-> Chỉ số chính Macro-F1 Score đạt đỉnh mới: {best_macro_f1:.4f}")

# 4. In ma trận nhầm lẫn tối hậu chính xác để Huy lấy số liệu viết báo cáo
print("\nMa trận nhầm lẫn chính thức (Khóa số liệu):")
print(confusion_matrix(y_true, final_preds))

print("\nBáo cáo hiệu năng chi tiết từng lớp:")
print(classification_report(y_true, final_preds, digits=4))

In [4]:
import numpy as np
import torch

# Số liệu mẫu thực tế của tập Train từ bộ dữ liệu UIT-VSFC của nhóm
# Nhãn 0 (Tiêu cực): 5268 mẫu | Nhãn 1 (Trung tính): 491 mẫu | Nhãn 2 (Tích cực): 5667 mẫu
train_counts = np.array([5268, 491, 5667])
total_samples = train_counts.sum()
num_classes = 3

# Tính toán trọng số chuẩn hóa theo công thức nghịch đảo phân bố
class_weights = total_samples / (num_classes * train_counts)

# Chuyển mảng trọng số sang dạng PyTorch Tensor và đẩy lên thiết bị GPU/CPU đang cấu hình
# Đảm bảo biến DEVICE (hoặc device) đã được Tài định nghĩa ở đầu tệp
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(DEVICE)

print("Mảng Class Weights huấn luyện tối ưu:", class_weights)

Mảng Class Weights huấn luyện tối ưu: [0.72298152 7.75695859 0.67207811]
